# RecordingNote 用 Colab バックエンド（Haas-ja-v1.0）

RecordingNote（https://recordingnote-beta-01.jiuzijiushan.chatgpt.site）の文字起こしを、
Gemini API の代わりに **自分の Google Colab 上の Haas-ja-v1.0** で行うためのノートブック。

## 位置づけ（重要）
- これは **個人版／自己接続用テンプレ**。自分のアカウントの Colab を、自分の RecordingNote から使う構成。
- 他人の録音を自分の Colab で処理する「中央提供」はしない。Colab の公式FAQ・追加規約では、
  ウェブサービスの提供・リモートプロキシ接続・第三者への Colab アクセス提供が禁止対象になっているため。
  配布版では、利用者各自がこのノートブックを自分のアカウントで起動し、自分の接続先URLを
  RecordingNote の個人設定に入れる形にする。
- Haas が担うのは **文字起こしだけ**。AIノート生成は RecordingNote 側の既存処理をそのまま使う。

## 使い方
1. ランタイム > ランタイムのタイプを変更 > **GPU (T4)** を選ぶ
2. セル1（確認・Driveマウント）→ セル2（導入）の順に実行する
3. セル2の表示どおり **セッションを再起動** し、セル1 → セル3（モデル読み込み）を再実行する
   （haas-asr 導入で protobuf が 5.x に巻き戻りモデル読み込みが壊れるため、6.33.5 へ戻す手順が必須。2026-11-11 実測）
4. セル4（APIサーバー＋トンネル＋Driveキュー）を実行すると、公開URLと API キーが表示され、
   Drive「講義録音/文字起こし/colab-backend.json」にも保存される
5. その URL とキーを RecordingNote の「個人バックエンド設定」（サイト側改修後）に入れる

## 仕様
- API: `GET /health`（認証なし）、`POST /jobs`（音声アップロード、要 X-API-Key）、`GET /jobs/{job_id}`（要 X-API-Key）
- 長尺は公式 long-form（無音点で分割してバッチ処理、batch_size=32 は T4 実測構成）
- Driveキュー併用: Drive「講義録音」に置かれた未処理音声も定期的に拾い、「文字起こし」へ .txt を出す（従来ノートブック互換）
- Colab は最大12時間・アイドル切断あり。切れたらこのノートブックを再実行する。ジョブ自体は Driveキュー側に残る

## 利用者自身の枠を使う流れ（配布版）
- Googleログインは **Colab側** で行う（このノートブックを開いた本人のアカウントの枠を使う）
- サイト側の「Googleでログイン」ボタンだけでは Colab は起動できない（消費者向けColabに、サイトからランタイムを起動する公式APIはない）。
  そのため接続はペアリング方式にする:
  1. RecordingNote で「自分のColabを接続」を押すと **ペアリングコード**（10分有効）が表示される
  2. このノートブックを自分のGoogleアカウントで開き、全セルを実行する
  3. 最後のセルでペアリングコードを入力すると、公開URLとAPIキーがサイトへ自動登録される
     （登録先URLが未設定の間は、表示されたURLとキーをサイトの個人設定へ手で貼る）
- PC（Linux/macOS）では公式 Colab CLI（google-colab-cli、OAuth認証）で起動を自動化する別経路もありうるが、Windows利用者には使えないため本線にはしない（配布対象はPC前提、iPadは対象外）


In [ ]:
# 1) GPU確認とDriveマウント、対象フォルダの準備
import subprocess
print(subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout or 'GPUなし: ランタイム > ランタイムのタイプを変更 > GPU を選んでください')
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
INPUT_DIR = Path('/content/drive/MyDrive/講義録音')
OUTPUT_DIR = INPUT_DIR / '文字起こし'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_EXT = {'.m4a','.mp3','.wav','.aac','.caf','.flac','.ogg','.webm'}
print('入力:', INPUT_DIR)
print('出力:', OUTPUT_DIR)


In [ ]:
# 2) 導入（Haas-ASR + APIサーバー一式）
# protobuf修正込み: haas-asr導入で protobuf が 5.x に巻き戻されモデル読み込みが壊れるため 6.33.5 へ戻す（2026-11-11実測で必須）
!pip -q install "haas-asr @ git+https://github.com/third-intelligence/Haas-ASR.git"
!pip -q install "protobuf==6.33.5"
!pip -q install fastapi uvicorn python-multipart pycloudflared
print('導入完了。ここで ランタイム > セッションを再起動 し、セル1 → セル3 の順に実行してください（このセルは再実行不要）')


In [ ]:
# 3) モデル読み込み（再起動後に実行）
import haas_asr
model = haas_asr.load_model('third-intelligence/Haas-ja-v1.0')
print('model loaded')


In [ ]:
# 4) 文字起こし関数 + Driveキュー（1回分）
import subprocess, threading
from pathlib import Path

VOCAB = {}  # 任意: {'考古学概説II': ['こうこがくがいせつ']} のように固有名詞を入れると認識が寄る
_vocabulary = haas_asr.Vocabulary(VOCAB) if VOCAB else None
_gpu_lock = threading.Lock()

def transcribe_file(src_path: str) -> str:
    """音声ファイルを 16kHz WAV 化して long-form 文字起こしする（GPUは直列に使う）"""
    wav = '/tmp/_rn_input.wav'
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(src_path),'-ar','16000','-ac','1',wav], check=True)
    audio = haas_asr.load_audio(wav)
    with _gpu_lock:
        if _vocabulary:
            return haas_asr.transcribe_longform(model, audio, batch_size=32, vocabulary=_vocabulary)
        return haas_asr.transcribe_longform(model, audio, batch_size=32)

def run_drive_queue_once():
    targets = [p for p in sorted(INPUT_DIR.iterdir())
               if p.suffix.lower() in AUDIO_EXT and not (OUTPUT_DIR / (p.stem + '.txt')).exists()]
    print('Driveキュー未処理:', [p.name for p in targets])
    for p in targets:
        print('処理中:', p.name, flush=True)
        text = transcribe_file(str(p))
        out = OUTPUT_DIR / (p.stem + '.txt')
        out.write_text(text + '\n', encoding='utf-8')
        print('保存:', out.name, f'({len(text)}文字)')
    return len(targets)


In [ ]:
# 5) APIサーバー起動 + Cloudflareクイックトンネル + Driveキュー巡回
import json, secrets, threading, time, uuid
from fastapi import FastAPI, File, UploadFile, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

# APIキー: Colabシークレット RECORDINGNOTE_API_KEY があればそれを使い、無ければ今回分を生成して表示する
try:
    from google.colab import userdata
    API_KEY = userdata.get('RECORDINGNOTE_API_KEY')
except Exception:
    API_KEY = None
if not API_KEY:
    API_KEY = secrets.token_urlsafe(32)

app = FastAPI(title='RecordingNote Haas backend')
app.add_middleware(CORSMiddleware,
    allow_origins=['https://recordingnote-beta-01.jiuzijiushan.chatgpt.site'],
    allow_methods=['GET','POST'], allow_headers=['*'])

jobs = {}

def check_key(x_api_key: str = Header(default='')):
    if x_api_key != API_KEY:
        raise HTTPException(status_code=401, detail='invalid api key')

@app.get('/health')
def health():
    return {'ok': True, 'engine': 'Haas-ja-v1.0', 'model_loaded': True}

@app.post('/jobs')
async def create_job(file: UploadFile = File(...), x_api_key: str = Header(default='')):
    check_key(x_api_key)
    job_id = uuid.uuid4().hex[:12]
    suffix = Path(file.filename or 'audio.webm').suffix or '.webm'
    work = Path(f'/tmp/jobs/{job_id}'); work.mkdir(parents=True, exist_ok=True)
    src = work / ('input' + suffix)
    src.write_bytes(await file.read())
    jobs[job_id] = {'status': 'queued', 'text': None, 'error': None}
    def run():
        jobs[job_id]['status'] = 'running'
        try:
            text = transcribe_file(str(src))
            out = OUTPUT_DIR / f'recordingnote-{job_id}.txt'
            out.write_text(text + '\n', encoding='utf-8')
            jobs[job_id].update(status='done', text=text, chars=len(text), output=str(out))
        except Exception as e:
            jobs[job_id].update(status='error', error=str(e))
    threading.Thread(target=run, daemon=True).start()
    return {'job_id': job_id, 'status': 'queued'}

@app.get('/jobs/{job_id}')
def get_job(job_id: str, x_api_key: str = Header(default='')):
    check_key(x_api_key)
    if job_id not in jobs:
        raise HTTPException(status_code=404, detail='job not found')
    return {'job_id': job_id, **jobs[job_id]}

def serve():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')
threading.Thread(target=serve, daemon=True).start()
time.sleep(3)

from pycloudflared import try_cloudflare
tunnel = try_cloudflare(port=8000)
PUBLIC_URL = tunnel.url
info = {'url': PUBLIC_URL, 'api_key': API_KEY, 'engine': 'Haas-ja-v1.0'}
(OUTPUT_DIR / 'colab-backend.json').write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
print('公開URL:', PUBLIC_URL)
print('APIキー (RecordingNoteの個人設定に入れる。第三者に共有しない):', API_KEY)
print('接続情報は Drive 講義録音/文字起こし/colab-backend.json にも保存した')

# Driveキュー巡回（60秒間隔、任意。止めたいときは ENABLE_DRIVE_QUEUE=False にして再実行）
ENABLE_DRIVE_QUEUE = True
def queue_loop():
    while ENABLE_DRIVE_QUEUE:
        try:
            run_drive_queue_once()
        except Exception as e:
            print('Driveキューでエラー:', e)
        time.sleep(60)
threading.Thread(target=queue_loop, daemon=True).start()
print('起動完了。/health で疎通確認できる')

# --- ペアリング登録（任意）---
# RecordingNote の「自分のColabを接続」で表示されたペアリングコードを入力すると、
# このバックエンドの公開URLとAPIキーをサイトへ登録する。
# SITE_REGISTER_URL が空の間は登録をスキップする（URLとキーをサイトの個人設定へ手で貼ること）。
SITE_REGISTER_URL = ''  # 例: 'https://recordingnote-beta-01.jiuzijiushan.chatgpt.site/api/colab/register'（サイト改修後に設定）
try:
    PAIRING_CODE = input('ペアリングコード（無ければそのままEnter）: ').strip()
except Exception:
    PAIRING_CODE = ''
if PAIRING_CODE and SITE_REGISTER_URL:
    import urllib.request
    req = urllib.request.Request(
        SITE_REGISTER_URL,
        data=json.dumps({'pairing_code': PAIRING_CODE, 'url': PUBLIC_URL, 'api_key': API_KEY}).encode(),
        headers={'Content-Type': 'application/json'})
    print('サイトへ登録:', urllib.request.urlopen(req, timeout=30).status)
elif PAIRING_CODE:
    print('登録先URLが未設定のため自動登録はスキップ。上の公開URLとAPIキーをサイトの個人設定に貼ってください')

